# ASL Fingerspelling (EMG) — Exploring One Random Subject

This notebook loads the **ASL Fingerspelling dataset for the Myo sensor** and walks through the data of **one randomly chosen subject**: what was recorded, the signal schema, EMG normalization, and visualizations of the EMG and IMU streams for the alphabet **A–Z**.

## What the dataset is

Nine users fingerspelled each letter of the alphabet while wearing **one Myo armband on the right forearm**. Each recording is ~5 s of streaming. The armband provides:

- **8 EMG channels** — surface electromyography, signed 8-bit ADC counts (−128..127), muscle activation.
- **IMU** — accelerometer (`AX,AY,AZ`, g), gyroscope (`GX,GY,GZ`, deg/s), orientation (`OR,OP,OY` = roll/pitch/yaw).

### File layout & schema
- Inside `alphabet_fingerspelling_dyfav.zip`: `dyfav/User1..User9/<id>_alphabet_<letter>_right.csv`.
- Each CSV is **one recording** of ~50 timesteps. **There is no header row and no counter column** — just **17 columns** in this fixed order:
  `EMG0..EMG7, AX, AY, AZ, GX, GY, GZ, OR, OP, OY`.
- The recording's **label** is the letter embedded in the filename (`..._alphabet_a_right.csv` -> `a`).
- Single arm only (`right`), so unlike the two-armband word dataset there is no `L`/`R` split.

> Runs on **Carnets Plus** (on-device iOS Jupyter): only the standard library + numpy/pandas/matplotlib are used, and CSVs are read **directly out of the zip** so nothing has to be extracted on the device.

_Source: Paudyal et al., "Sceptre" (ACM IUI 2016) & "DyFAV" (ACM IUI 2017), Arizona State University. Mendeley dataset `dbymbhhpk9`, CC BY 4.0._

## 1. Setup

Put `alphabet_fingerspelling_dyfav.zip` (or its `dyfav/` folder) somewhere the notebook can find it — the same folder as this notebook works. The loader searches the current folder and a few parents.

In [ ]:
import io, os, re, zipfile, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Column order is fixed and the CSVs have no header.
EMG = [f'EMG{i}' for i in range(8)]
COLS = EMG + ['AX', 'AY', 'AZ', 'GX', 'GY', 'GZ', 'OR', 'OP', 'OY']

# Set a seed for a reproducible 'random' subject; comment out for a fresh pick each run.
SEED = None
if SEED is not None:
    random.seed(SEED)

## 2. Locate and open the data

We open the zip in memory via the stdlib `zipfile` — no files are written to disk. If you've already extracted the `dyfav/` folder, that's used instead.

In [ ]:
def _find(names, max_up=4):
    """Search cwd and up to `max_up` parent dirs (recursively) for the first matching filename."""
    root = os.getcwd()
    for _ in range(max_up + 1):
        for dirpath, _dirs, files in os.walk(root):
            for n in names:
                if n in files:
                    return os.path.join(dirpath, n)
        root = os.path.dirname(root)
    return None

def _find_dir(name, max_up=4):
    root = os.getcwd()
    for _ in range(max_up + 1):
        for dp, dirs, _f in os.walk(root):
            if os.path.basename(dp) == name:
                return dp
        root = os.path.dirname(root)
    return None

ZIP_PATH = _find(['alphabet_fingerspelling_dyfav.zip'])
ZF = zipfile.ZipFile(ZIP_PATH) if ZIP_PATH else None

def list_csvs():
    if ZF is not None:
        return [n for n in ZF.namelist() if n.endswith('.csv')]
    base = _find_dir('dyfav')
    if not base:
        raise FileNotFoundError('Place alphabet_fingerspelling_dyfav.zip next to this notebook.')
    out = []
    for dp, _d, files in os.walk(base):
        out += [os.path.join(dp, f) for f in files if f.endswith('.csv')]
    return out

def read_bytes(path):
    return ZF.read(path) if ZF is not None else open(path, 'rb').read()

print('Data source:', ZIP_PATH or _find_dir('dyfav'))

## 3. Build an index of every recording

Each path looks like `dyfav/User1/291982754_alphabet_a_right.csv`. We parse out **user** and **letter** for every file.

In [ ]:
def parse(path):
    parts = path.replace('\\', '/').split('/')
    user = next((p for p in parts if p.lower().startswith('user')), parts[-2])
    m = re.search(r'alphabet_([a-z]+)_', os.path.basename(path).lower())
    letter = m.group(1) if m else '?'
    return user, letter

rows = []
for p in list_csvs():
    user, letter = parse(p)
    rows.append({'user': user, 'letter': letter, 'path': p})

index = pd.DataFrame(rows)
print('Total recordings:', len(index))
print('Subjects:', index['user'].nunique(), '| Letters:', index['letter'].nunique())
index.head()

## 4. Pick one random subject

A *subject* is one user. We pick one at random and keep only its recordings.

In [ ]:
chosen_user = random.choice(sorted(index['user'].unique()))
subj = index[index['user'] == chosen_user].reset_index(drop=True)

print('Chosen subject :', chosen_user)
print('Recordings     :', len(subj))
print('Letters covered:', subj['letter'].nunique(), '/ 26')
subj['letter'].value_counts().sort_index()

## 5. A small CSV reader

Reads one recording straight from the zip into a DataFrame with the fixed column names. `drop_warmup` removes the first rows where **all EMG channels are 0** — the Myo EMG needs a sample or two to start streaming while the IMU is already valid.

In [ ]:
def read_recording(path, drop_warmup=True):
    df = pd.read_csv(io.BytesIO(read_bytes(path)), header=None, names=COLS)
    if drop_warmup:
        df = df[(df[EMG] != 0).any(axis=1)].reset_index(drop=True)
    return df

example = subj.iloc[random.randrange(len(subj))]
df = read_recording(example['path'])
print('Example recording:', os.path.basename(example['path']))
print('Letter          :', example['letter'].upper())
print('Shape after warm-up drop:', df.shape)
df.head()

## 6. EMG — the 8 raw channels for one recording

The 8 EMG channels over the recording, **as stored** (signed int8, bipolar). EMG is roughly zero-mean noise that bursts when the underlying muscles contract, so the interesting structure is the **amplitude envelope** across the gesture — which is exactly what the normalization below extracts.

In [ ]:
t = np.arange(len(df))
fig, axes = plt.subplots(8, 1, figsize=(10, 11), sharex=True)
for i in range(8):
    axes[i].plot(t, df[f'EMG{i}'], lw=0.9, color='C0')
    axes[i].axhline(0, color='k', lw=0.4)
    axes[i].set_ylabel(f'EMG{i}')
axes[-1].set_xlabel('timestep')
fig.suptitle(f"Raw EMG — {chosen_user}, letter '{example['letter'].upper()}'", y=0.995)
fig.tight_layout()
plt.show()

## 7. IMU — motion of the armband

The accelerometer, gyroscope, and orientation streams describe how the forearm moved during the sign. Unlike EMG these are smooth, low-frequency trajectories.

In [ ]:
groups = [('Accelerometer (g)', ['AX', 'AY', 'AZ']),
          ('Gyroscope (deg/s)', ['GX', 'GY', 'GZ']),
          ('Orientation (roll/pitch/yaw)', ['OR', 'OP', 'OY'])]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, (title, base) in zip(axes, groups):
    for b in base:
        ax.plot(t, df[b], lw=1.1, label=b)
    ax.set_ylabel(title, fontsize=9)
    ax.legend(fontsize=8, ncol=3, loc='upper right')
axes[-1].set_xlabel('timestep')
fig.suptitle(f"IMU — {chosen_user}, letter '{example['letter'].upper()}'", y=0.995)
fig.tight_layout()
plt.show()

## 8. EMG amplitude normalization (per-subject, %-of-peak)

This dataset has **no maximum-voluntary-contraction (MVC) trials**, so true %MVC isn't computable. Instead we use the standard substitute: **peak / sub-maximal normalization**, computed **per channel, per subject** (electrode placement and skin differ between people — that's the cross-subject comparability real MVC provides).

**Order matters.** Amplitude normalization belongs *after* rectification, because %-of-peak means *fraction of the maximum activation level*, and activation level is the rectified amplitude. The reference is the **99th percentile** of `|EMG|` (not the raw max — that way one stray spike can't set the scale; Myo int8 saturation here is ~0.005%, but the percentile is robust regardless). The reference is computed on the **same representation we normalize** (rectified samples), which is the one step that genuinely does *not* commute.

Pipeline: `rectify (|x|) -> per-subject 99th-pct reference -> divide & clip to [0, 1]`. (Myo EMG is already hardware band-pass filtered on-device, so no user filter is needed before rectifying.)

In [ ]:
def subject_emg_reference(subj_df, pct=99):
    """Per-channel reference = pct-percentile of |EMG| over ALL of one subject's recordings."""
    stacked = np.concatenate([read_recording(p)[EMG].abs().values for p in subj_df['path']])
    return np.percentile(stacked, pct, axis=0)            # shape (8,)

def normalize_emg(df, ref):
    """Rectify and scale EMG to [0, 1] as a fraction of the subject's per-channel peak."""
    out = df.copy()
    out[EMG] = (df[EMG].abs().values / ref).clip(0, 1)
    return out

def rms_envelope(x, win=5):
    """Simple moving-RMS for display smoothing. ponytail: naive same-length convolution."""
    x = np.asarray(x, dtype=float)
    return np.sqrt(np.convolve(x ** 2, np.ones(win) / win, mode='same'))

REF = subject_emg_reference(subj)
pd.Series(REF, index=EMG, name='99th-pct |EMG| (reference)')

## 9. Raw vs normalized — one channel

The most-active channel in the example recording, shown raw (bipolar int8) and after rectify + normalize. The red line is a moving-RMS envelope of the normalized signal — a clean activation curve in `[0, 1]` that's comparable across channels and subjects.

In [ ]:
ndf = normalize_emg(df, REF)
ch = EMG[int(np.argmax(df[EMG].abs().mean().values))]   # busiest channel

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, df[ch], lw=0.9, color='C7')
ax[0].axhline(0, color='k', lw=0.4)
ax[0].set_ylabel(f'{ch} raw (int8)')
ax[0].set_title(f"{chosen_user}, letter '{example['letter'].upper()}' — channel {ch}")
ax[1].plot(t, ndf[ch], lw=0.9, color='C0', label='|EMG| / ref')
ax[1].plot(t, rms_envelope(ndf[ch]), lw=2.0, color='C3', label='RMS envelope')
ax[1].set_ylim(0, 1.05)
ax[1].set_ylabel(f'{ch} normalized')
ax[1].set_xlabel('timestep')
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 10. Which muscles fire for which letter? (normalized)

For every recording we take the **mean normalized EMG** per channel, then average across repetitions of each letter. Because values are now `%-of-peak`, rows are comparable across channels **and** across subjects — a muscle-activation fingerprint per letter. Distinct rows mean the letters are EMG-separable, which is what a classifier exploits.

In [ ]:
per_letter = {}
for letter, grp in subj.groupby('letter'):
    mats = [normalize_emg(read_recording(p), REF)[EMG].mean().values for p in grp['path']]
    mats = [m for m in mats if not np.isnan(m).all()]
    if mats:
        per_letter[letter] = np.mean(mats, axis=0)

act = pd.DataFrame(per_letter, index=EMG).T.sort_index()   # rows=letters, cols=8 channels

fig, ax = plt.subplots(figsize=(8, 0.34 * len(act) + 2))
im = ax.imshow(act.values, aspect='auto', cmap='viridis', vmin=0)
ax.set_xticks(range(len(EMG)))
ax.set_xticklabels(EMG, rotation=90, fontsize=8)
ax.set_yticks(range(len(act)))
ax.set_yticklabels([l.upper() for l in act.index], fontsize=8)
ax.set_xlabel('EMG channel')
ax.set_title(f'Mean normalized EMG per letter — {chosen_user}')
fig.colorbar(im, ax=ax, label='mean normalized EMG (frac of peak)')
fig.tight_layout()
plt.show()

## 11. Repetition consistency for one letter (normalized)

How repeatable is a single letter? We overlay the normalized EMG envelope (RMS across all 8 channels at each timestep) for every repetition of one letter. Tight overlap = consistent execution; spread = natural variability the model must tolerate. Recordings differ in length, so the x-axis is each rep's own timestep.

In [ ]:
target = subj['letter'].value_counts().idxmax()   # the letter with the most reps
reps = subj[subj['letter'] == target]['path'].tolist()

fig, ax = plt.subplots(figsize=(10, 4))
for j, p in enumerate(reps):
    d = normalize_emg(read_recording(p), REF)
    env = np.sqrt((d[EMG] ** 2).mean(axis=1))   # RMS over 8 normalized channels
    ax.plot(np.arange(len(env)), env, lw=1.3, label=f'rep {j + 1}')
ax.set_xlabel('timestep')
ax.set_ylabel('RMS normalized EMG across 8 channels')
ax.set_title(f"Repetitions of letter '{target.upper()}' — {chosen_user}")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Notes

- **Normalization is a surrogate, not true %MVC:** there are no MVC trials, so amplitudes are scaled to each subject's own 99th-percentile peak. It assumes each subject reaches near-peak activation somewhere across 26 letters × ~5 reps — reasonable, but an assumption real MVC wouldn't need. Absolute values are not anchored to a physiological maximum.
- **Per-subject:** recompute `REF` per subject (`subject_emg_reference`). Do not reuse one subject's reference for another.
- **Reproducibility:** set `SEED` in cell 1 to always land on the same subject.
- **Warm-up rows:** `read_recording(..., drop_warmup=False)` keeps the leading all-zero-EMG rows.
- **No counter column:** the time axis is the row index; recordings vary slightly in length (~50 steps).
- **Single arm:** all data is the right forearm; there is no left-arm signal here.
- **Carnets:** everything above uses only stdlib + numpy/pandas/matplotlib, all bundled. No extraction, no extra installs.